# Polariton condensate: dynamics, thresholds and vortices

One notebook for the 0D condensate–reservoir model and 2D Gaussian/ring pumping.
Units: µm, ps, meV. Default cells load portable results and run a short 64² example.
Full simulations are opt-in through `DO_*` flags.

[English report](latex/report_en.pdf) · [Russian report](latex/report.pdf) ·
[Methods and figure parameters](docs/methods.md) · [Recorded checks](docs/diagnostic_summary.md)

## 1. Configuration and methods

0D dynamics use adaptive DOP853. The 2D solver uses FFT kinetic half-steps and a coupled local RK4 step (Strang splitting; second order overall).

In [ ]:
from pathlib import Path
import json
from IPython.display import display, Image
from numerics import load_config, check_config, simulate_2d
from model import parameters, make_grid
from plotting import load_report_case, make_report

config = load_config("validation")
check_config(config)
p = parameters(config)
print("K [meV µm²]:", p["K"], "homogeneous threshold [µm⁻² ps⁻¹]:", p["Pth0D"])

## 2. Saved results

The spot reaches a steady state at 500 ps. The ring forms a vortex with charge q = −3 at 2000 ps for the recorded initial seed. These images and fields are included in Git.

In [ ]:
figure_dir = Path("results/report_figures")
for name in ("05_spatial_profiles.png", "09_ring_transient_evolution.png"):
    display(Image(filename=str(figure_dir / name), width=850))
print("All nine PNGs and both GIFs:", figure_dir)

In [ ]:
from numerics import vortex_diagnostics
ring = load_report_case("ring")
vortex = vortex_diagnostics(ring["psi"], make_grid(ring["config"]), ring["config"])
print("Ring time [ps]:", ring["time_ps"])
print("Charge q:", vortex["q"])
print("Lz / (ħ NC):", vortex["Lz_over_hbar_NC"])

## 3. Short simulation

This 64², 2 ps run demonstrates field–reservoir evolution and saving. It is not a production result or a stationarity test. New files go to the ignored `results/runs/` directory.

In [ ]:
demo = simulate_2d(load_config("validation"))
print("Saved run:", demo["path"])
print("Status:", demo["summary"]["run_status"])

## 4. Recorded numerical checks

The included metadata records the original convergence and stationarity checks. The optional cell below reruns the existing small test suite.

In [ ]:
metadata = json.loads(Path("results/analysis_data/report_metadata.json").read_text())
for name in ("gaussian", "ring"):
    case = metadata["cases"][name]
    print(name, case["time_ps"], "ps:", case["summary"]["stationarity"]["stationarity_status"])
print("Final ring checks:", metadata["checks"]["final_ring_checks"]["status"])

In [ ]:
DO_CHECKS = False
if DO_CHECKS:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider"], check=True)

## 5. Redraw the report figures

This reads the included NPZ and JSON only. It performs no 0D/2D time integration and needs no original run directories. The preview directory keeps the committed PNGs unchanged.

In [ ]:
DO_REDRAW = False
if DO_REDRAW:
    figures = make_report(output_dir="results/preview")
    print("Saved", len(figures), "figures to results/preview")

## 6. Full simulations (optional)

`report_gaussian` and `report_ring` contain the exact configurations used for the reported final fields. Both use 512² points in a 256 × 256 µm² domain and the same continuous random seed. The spot uses Δt = 0.025 ps to 500 ps; the ring uses Δt = 0.05 ps to 2000 ps. These runs take much longer than the demo.

In [ ]:
DO_REPORT_SPOT = False
DO_REPORT_RING = False
for enabled, profile in ((DO_REPORT_SPOT, "report_gaussian"), (DO_REPORT_RING, "report_ring")):
    if enabled:
        run = simulate_2d(load_config(profile))
        print(profile, run["path"], run["summary"]["run_status"])

### Threshold scans

These opt-in calculations find the instability threshold of the condensate-free state. They create new runs and may require long propagation to separate leading modes.

In [ ]:
DO_THRESHOLDS = False
if DO_THRESHOLDS:
    from numerics import find_threshold_2d
    for profile in ("report_gaussian", "report_ring"):
        threshold = find_threshold_2d(load_config(profile))
        print(profile, threshold)

### Resume a saved run

Set a local run directory to continue from its checkpoint. The configuration must remain compatible with the saved state.

In [ ]:
RUN_TO_RESUME = None
if RUN_TO_RESUME is not None:
    from numerics import load_run
    saved = load_run(RUN_TO_RESUME)
    resumed = simulate_2d(saved["config"], resume_dir=RUN_TO_RESUME)
    print(resumed["path"], resumed["summary"]["run_status"])

### Original research archive

The full grid/time convergence series and detailed transient histories occupy about 6 GB in local `results/runs/`. The completed comparison values are included in `report_metadata.json`. Repeating the historical archive-dependent workflows requires those original runs; the notebook preview, demo and nine-figure export work without them. See `docs/methods.md` for protocols and the report for physical interpretation.